# 05 - Model Validation

What the dissolved-oxygen model is worth, measured against baselines that do
not need a model at all (`aquanexus.ml.validator`, Phase 2c).

The synthetic-HSI model is not validated here. Its labels are generated, so
there is no baseline to beat and no external truth to fail against - the check
that matters for it is the falsification test in `02_data_exploration.ipynb`.

Everything below is grouped cross-validation with **whole stations held out**.

In [ ]:
import sys; sys.path.insert(0, '../src')
import glob

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from aquanexus.config import settings
from aquanexus.data.dataset import (DO_FEATURES, DO_RANGE, build_water_quality_dataset,
                                    interpolate_hydraulics)
from aquanexus.data.loader import filter_stations, load_many
from aquanexus.ml.evaluator import residual_summary
from aquanexus.ml.models import HabitatPredictor, ModelConfig
from aquanexus.ml.validator import ModelValidator

pd.set_option('display.width', 200)

In [ ]:
files = sorted(glob.glob(str(settings.RAW_DIR / 'waterquality' / 'saitama_*.xlsx')))
observations = filter_stations(load_many(files), water_body=settings.RIVER_NAME_JA)
sweep = pd.read_csv(settings.PROCESSED_DIR / 'ayase_flow_sweep.csv')

water = build_water_quality_dataset(observations, sweep)
do_features = [f for f in DO_FEATURES if f in water.columns]
config = ModelConfig(model_type='linear', output_range=DO_RANGE)

validator = ModelValidator(water, do_features, 'dissolved_oxygen', 'station')
print(f'{len(water)} observations, {water.station.nunique()} stations, '
      f'{water.timestamp.min():%Y-%m} to {water.timestamp.max():%Y-%m}')

## 1. The comparison table

Three baselines, as ML_STRATEGY §7.3 specifies, with one substitution forced by
the data: the spec's "previous day's score" becomes the **previous sample at
the same station**, because sampling is monthly. That is the honest version,
and for a slowly varying quantity it is a strong baseline rather than a foil.

In [ ]:
report = validator.run(model_types=('linear', 'random_forest', 'xgboost'),
                       output_range=DO_RANGE)
report.table().round(3)

In [ ]:
for note in report.notes:
    print(f'- {note}')

## 2. The result that should temper everything else

"Same dissolved oxygen as last month at this station" is a competitive model.

In [ ]:
model_row = next(m for m in report.results if m.model == 'linear')
persistence_row = next(m for m in report.results if m.model == 'persistence')

comparison = pd.DataFrame([model_row.as_row(), persistence_row.as_row()])[
    ['model', 'n', 'rmse', 'mae', 'r2']]
print(comparison.round(3).to_string(index=False))
print(f'\nmodel advantage : {persistence_row.rmse - model_row.rmse:+.3f} mg/L RMSE, '
      f'{model_row.r2 - persistence_row.r2:+.3f} R2')
print(f'model deficit   : {model_row.mae - persistence_row.mae:+.3f} mg/L MAE '
      '(persistence is better here)')

The model wins on RMSE and R2 by a small margin and **loses on MAE**. It is
better at avoiding large errors and slightly worse on the typical one.

A comparison table without persistence in it would have made an R2 of 0.44 look
like the model had learned the river. What it has mostly learned is that
dissolved oxygen does not change much between monthly samples.

That is not a reason to discard it - the model generalises to stations it has
never seen, which persistence cannot do at all, and it accepts hypothetical
states, which is what the scenario endpoint needs. It is a reason not to
oversell it.

In [ ]:
labels = ['Ridge\n(shipped)', 'persistence', 'random\nforest', 'xgboost',
          'mean\n(floor)', 'hydraulic\nonly']
scores = {m.model: m for m in report.results}
order = ['linear', 'persistence', 'random_forest', 'xgboost', 'mean', 'hydraulic-only']
colours = ['#c0392b', '#7f8c8d', '#2e86ab', '#2e86ab', '#bdc3c7', '#bdc3c7']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 3.4))
ax1.bar(labels, [scores[k].rmse for k in order], color=colours)
ax1.set_ylabel('RMSE (mg/L) - lower is better')
ax1.set_title('Grouped CV, each station held out', loc='left', fontweight='bold')
ax2.bar(labels, [scores[k].mae for k in order], color=colours)
ax2.set_ylabel('MAE (mg/L) - lower is better')
ax2.set_title('On typical error, persistence wins', loc='left', fontweight='bold')
plt.tight_layout()

## 3. Spatial validation - per station

ML_STRATEGY §5.2 in the form this data supports: every station is held out in
turn and scored on its own, so one badly predicted station cannot hide inside a
pooled mean.

In [ ]:
validator.by_group(config).round(3)

## 4. Event-based validation, reframed

The spec asks for named flood and drought events. There are none in a monthly
grab-sample record, so the tails of the drivers stand in for them: does the
model hold together at the extremes of discharge and temperature?

In [ ]:
by_flow = validator.by_condition(config, 'discharge')
by_temperature = validator.by_condition(config, 'water_temp')
print(by_flow.round(3).to_string(index=False))
print()
print(by_temperature.round(3).to_string(index=False))

### The known failure, quantified

At low flow the model under-predicts oxygen by around 2 mg/L - a bias an order
of magnitude larger than in the middle band, and in the wrong direction for
safety: it says the water is worse than it is. **Drought is exactly when an
oxygen diagnosis matters**, so the model is least reliable precisely where it
would be used.

The API attaches this as a caveat to every dissolved-oxygen response rather
than leaving the caller to discover it.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.3))
bands = by_flow.band
palette = ['#c0392b', '#bdc3c7', '#2e86ab']
ax1.bar(bands, by_flow.rmse, color=palette)
ax1.set_ylabel('RMSE (mg/L)')
ax1.set_title('Error by flow regime', loc='left', fontweight='bold')
ax2.bar(bands, by_flow.bias, color=palette)
ax2.axhline(0, color='#1b2a41', lw=1)
ax2.set_ylabel('bias (mg/L)')
ax2.set_title('Under-predicts oxygen at low flow', loc='left', fontweight='bold')
for ax in (ax1, ax2):
    ax.set_xticks(range(len(bands)))
    ax.set_xticklabels([f'{b}\n({q:.1f} m3/s)' for b, q in
                        zip(bands, by_flow.discharge_mean, strict=True)], fontsize=8)
plt.tight_layout()

## 5. Residuals

Where the errors sit across the range of the observation itself.

In [ ]:
predictions = validator.cross_val_predict(config)
residual_summary(water.dissolved_oxygen, predictions)

In [ ]:
residual = predictions - water.dissolved_oxygen.to_numpy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.scatter(water.dissolved_oxygen, predictions, s=22, alpha=0.75,
            color='#2e86ab', edgecolor='white', lw=0.4)
limits = [water.dissolved_oxygen.min() - 0.5, water.dissolved_oxygen.max() + 0.5]
ax1.plot(limits, limits, color='#1b2a41', lw=1.2)
ax1.set_xlabel('observed DO (mg/L)')
ax1.set_ylabel('out-of-fold prediction (mg/L)')
ax1.set_title('Regression to the mean at both ends', loc='left', fontweight='bold')

ax2.scatter(water.discharge, residual, s=22, alpha=0.75, color='#c0392b',
            edgecolor='white', lw=0.4)
ax2.axhline(0, color='#1b2a41', lw=1)
ax2.set_xscale('log')
ax2.set_xlabel('discharge (m3/s, log)')
ax2.set_ylabel('prediction - observed (mg/L)')
ax2.set_title('Residual against flow', loc='left', fontweight='bold')
plt.tight_layout()

print(f'observed range {water.dissolved_oxygen.min():.1f}-'
      f'{water.dissolved_oxygen.max():.1f} mg/L, '
      f'predicted range {predictions.min():.1f}-{predictions.max():.1f} mg/L')

The predicted range is much narrower than the observed one. A model that cannot
reach 3 mg/L cannot flag the hypoxic events that a habitat diagnosis exists to
catch, and at n=138 with four stations there is no amount of tuning that fixes
that - it needs more data, particularly at low flow.

## 6. What the low-flow bias does to the scenario endpoint

`/scenario_run` moves the model's inputs; it does not re-run HEC-RAS. Reducing
discharge by 60% therefore leaves depth, velocity and width at their baseline
values - a state the river cannot physically be in - and the model, which
learned a negative discharge coefficient, **raises** predicted oxygen.

In [ ]:
model = HabitatPredictor.load(settings.MODELS_DIR / 'dissolved_oxygen_v1.joblib')
baseline = water[do_features].median().to_frame().T

drought_api = baseline.copy()
drought_api['discharge'] = baseline.discharge.iloc[0] * 0.4

print(f'baseline discharge {baseline.discharge.iloc[0]:.2f} m3/s  ->  '
      f'DO {model.predict(baseline)[0]:.2f} mg/L')
print(f'-60% discharge, hydraulics untouched (what the API does)  ->  '
      f'DO {model.predict(drought_api)[0]:.2f} mg/L')

Re-interpolating the reach hydraulics from the HEC-RAS sweep at the reduced
discharge - the physically consistent version - moves depth, velocity and width
with it:

In [ ]:
drought_physical = drought_api.copy()
hydraulics = interpolate_hydraulics(sweep, float(drought_api.discharge.iloc[0]))
drought_physical['reach_depth'] = hydraulics.depth.mean()
drought_physical['reach_velocity'] = hydraulics.velocity.mean()
drought_physical['reach_top_width'] = hydraulics.top_width.mean()
drought_physical['reach_froude'] = (
    hydraulics.velocity.mean() / np.sqrt(9.80665 * hydraulics.depth.mean()))

print(f'-60% discharge, hydraulics re-interpolated  ->  '
      f'DO {model.predict(drought_physical)[0]:.2f} mg/L')
print()
print(pd.concat([baseline, drought_api, drought_physical],
                keys=['baseline', 'API scenario', 'physical'])
      .droplevel(1)[['discharge', 'reach_depth', 'reach_velocity',
                     'reach_top_width', 'reach_froude']].round(3))

Making the inputs physically coherent moves the answer **further up**, so the
direction is not a feature-consistency bug. It comes from the training data:
dissolved oxygen and discharge are negatively associated here, and not only
across stations.

In [ ]:
correlations = water.groupby('station').apply(
    lambda block: pd.Series({
        'n': len(block),
        'mean_discharge': block.discharge.mean(),
        'mean_do': block.dissolved_oxygen.mean(),
        'corr(discharge, DO) within station': block.discharge.corr(block.dissolved_oxygen),
    }), include_groups=False)
print(f'pooled corr(discharge, DO): '
      f'{water.discharge.corr(water.dissolved_oxygen):+.3f}')
correlations.round(3)

The association is negative at **every** station, from -0.14 to -0.60. In this
river that is readable: high flow arrives with storm and combined-sewer load,
which consumes oxygen faster than the extra turbulence replaces it.

So "less flow, more oxygen" is what the record says, and the model is
reproducing it rather than inventing it. The DEVLOG entry calling the scenario
response *physically wrong* was too strong - it is contrary to a
reaeration-only expectation, not to this data.

**It still should not be believed at low flow**, for three reasons that are
about support rather than sign:

1. Section 4: the model is biased -2 mg/L in the low-flow band.
2. The low-flow samples are mostly one shallow upstream station (55畷橋, mean
   2.5 m3/s), so "low flow" and "that station" are partly the same variable.
3. What makes a real drought dangerous - higher temperature, longer residence
   time, concentrated oxygen demand - is not in this feature set, and a monthly
   grab-sample record of a mild low-flow range cannot show it.

Two separate fixes follow, and neither is a patch to the response text:
re-derive hydraulics from the sweep inside `/scenario_run` so the state is at
least coherent, and get low-flow observations before trusting any drought
answer. Until then the API caveat stands.

## 7. Debt register

Everything known to be wrong or unfinished that touches these results. Kept
here as well as in `DEVLOG.md` so it travels with the analysis.

In [ ]:
debt = pd.DataFrame([
    {'item': '4 cross-sections cut through bank (RS 12500/14000/18000/24000)',
     'bites': 'reach-mean hydraulics -> every hydraulic feature',
     'status': 'EXCLUDED 2026-09-08; worth 64% of RMSE, see docs/'},
    {'item': 'Manning n uncalibrated (no gauged rating curve for this reach)',
     'bites': 'depth and velocity carry systematic error',
     'status': 'quantified at ~15% of RMSE; still uncalibrated'},
    {'item': 'Scenario endpoint unreliable below ~2 m3/s',
     'bites': '/scenario_run answers, shown in section 6',
     'status': 'caveated in API, not fixed'},
    {'item': '/scenario_run held hydraulics fixed when discharge changed',
     'bites': 'every discharge scenario described an impossible state',
     'status': 'FIXED 2026-09-08 - re-interpolated from the sweep'},
    {'item': 'n=138 observations from 4 stations, monthly grab samples',
     'bites': 'every metric in this notebook',
     'status': 'inherent to the data source'},
    {'item': 'Model beats persistence by 0.009 R2, loses on MAE',
     'bites': 'the headline claim',
     'status': 'reported, section 2'},
    {'item': 'temp x discharge synergy (-1.02 mg/L) was one station only',
     'bites': 'README / ML_METHODOLOGY Phase 2b claim',
     'status': 'corrected in 04_explainability.ipynb'},
    {'item': 'HSI labels are synthetic',
     'bites': 'the habitat model and everything served from it',
     'status': 'labelled everywhere, falsification-tested only'},
])
debt

## Verdict

The dissolved-oxygen model is a **defensible small result**: it predicts a real
measurement on stations it has not seen, it beats a naive floor, and the
hydraulic model contributes a measurable if modest part of that. It does not
beat "same as last month" by much, it cannot reach the extremes, and it is
least trustworthy at low flow.

Stated that way it is worth showing. Stated as "R2 0.99 habitat prediction" it
would not be.